# Lesson 4.2 — Custom Agent Loops, Harnesses, and Memory

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Build a tool-use agent loop as a reusable harness
- Add short-term memory and a step budget to prevent runaways
- Manage the context window inside the loop (pruning/compaction)
- Expose a clean stop condition and error handling

**You will produce:** A reusable AgentHarness class with memory, a step cap, and tool dispatch (offline-safe).

## Concepts

A **custom agent loop** (harness) is the engine behind an agent: it repeatedly calls Claude, executes requested tools, feeds results back, and stops when the goal is met or a budget is hit.

Essential elements:
- **Tool dispatch**: map `tool_use` blocks to your functions (Phase 3).
- **Memory**: keep a working record (recent turns, key facts) and compact it to avoid bloat (Phase 2).
- **Step budget / stop condition**: cap iterations to prevent infinite loops and runaway cost.
- **Error handling**: catch tool and API errors and decide retry vs abort (Phase 5).

This is the self-hosted alternative to the **Claude Agent SDK** (Lesson 4.3), giving you full control over the harness.

> **Reference:** [Anthropic - Agent SDK overview](https://docs.anthropic.com/en/docs/agents-and-tools/agent-sdk/overview)

In [ ]:
# !pip install anthropic
import os

try:
    import anthropic
except ImportError:
    anthropic = None
    print("anthropic SDK not installed. Run: pip install anthropic")

API_KEY = os.environ.get("ANTHROPIC_API_KEY")
HAVE_API = bool(API_KEY) and anthropic is not None

client = anthropic.Anthropic() if HAVE_API else None  # reads ANTHROPIC_API_KEY

# Current model aliases (confirm latest at https://docs.anthropic.com/en/docs/about-claude/models)
MODEL_HAIKU = "claude-haiku-4-5"
MODEL_SONNET = "claude-sonnet-4-5"
MODEL_OPUS = "claude-opus-4-1"

def mock_text(prompt):
    """Offline fallback so the notebook runs without an API key."""
    return f"[MOCK RESPONSE] Set ANTHROPIC_API_KEY for a live call. Prompt was: {prompt[:80]}..."

print("API ready" if HAVE_API else "No API key - cells will print mock responses")

## Code-Along: Step 1 — A harness skeleton

Loop with a step cap and tool dispatch; fully runnable offline.

In [ ]:
class AgentHarness:
    def __init__(self, tools, dispatch, model=None, max_steps=5):
        self.tools = tools
        self.dispatch = dispatch  # name,input -> result string
        self.model = model or MODEL_SONNET
        self.max_steps = max_steps
        self.memory = []  # working memory / message history

    def run(self, goal):
        self.memory.append({"role": "user", "content": goal})
        for step in range(self.max_steps):
            msg = self._call()
            if not HAVE_API:
                return f"[MOCK] completed '{goal}' in 1 step"
            if msg.stop_reason != "tool_use":
                return msg.content[0].text  # stop condition met
            self.memory.append({"role": "assistant", "content": msg.content})
            results = []
            for b in msg.content:
                if b.type == "tool_use":
                    out = self.dispatch(b.name, b.input)
                    results.append({"type": "tool_result",
                        "tool_use_id": b.id, "content": str(out)})
            self.memory.append({"role": "user", "content": results})
        return "[stopped: step budget exhausted]"

    def _call(self):
        if not HAVE_API:
            return None
        return client.messages.create(model=self.model, max_tokens=600,
            tools=self.tools, messages=self.memory)

print("AgentHarness defined")

## Code-Along: Step 2 — Run the harness with a tool

Reuse the calculator pattern from Phase 3.

In [ ]:
CALC = {"name": "calculator", "description": "do math",
    "input_schema": {"type": "object",
        "properties": {"expression": {"type": "string"}}, "required": ["expression"]}}

def dispatch(name, inp):
    if name == "calculator":
        try:
            return eval(inp["expression"], {"__builtins__": {}}, {})
        except Exception as e:
            return f"error: {e}"
    return "unknown tool"

agent = AgentHarness([CALC], dispatch, max_steps=4)
print(agent.run("What is 15 * 7 + 2?"))

## Exercise

Add a `compact_memory()` method to `AgentHarness` that, once memory exceeds N turns, replaces the oldest turns with a single summary string (reuse the Phase 2 compaction idea).

In [ ]:
# TODO: def compact_memory(self, keep_recent=4):
#   - if len(self.memory) <= keep_recent: return
#   - summarize older turns into one note and splice it in
# TODO: call compact_memory() at the top of each run() iteration

## Key Takeaways

- An agent harness loops: call Claude, dispatch tools, feed results back, stop at goal/budget.
- Always cap steps to prevent infinite loops and runaway cost.
- Maintain and compact memory to keep the context window healthy.
- A custom loop is the self-hosted alternative to the Claude Agent SDK.

---

**Next:** [Lesson 4.3 — The Claude Agent SDK, Hosting Models, and Hooks](./lesson-4.3-agent-sdk-and-hosting.ipynb)